In [68]:
import re
import time
import ollama

# Configure your local Qwen model identifier
OLLAMA_MODEL_NAME = "gemma3:4b-it-qat"


def is_structural_table(text: str) -> bool:
  """Deterministic detector for tabular fragments (handles markdown and Docling formats)."""
  # Check for standard markdown table syntax
  has_markdown_table = text.count("|") >= 4 and (
      (":---" in text) or ("---:" in text)
  )
  return has_markdown_table


def predict_dynamic_k(
    chunk_text: str, model_name: str = OLLAMA_MODEL_NAME
) -> int:
  """Evaluates chunk semantic scope across k in {0, 1, 2, 3} using full-chunk analysis."""
  # ==========================================================================
  # Tier 1: Deterministic Syntax Pre-Gating (Host CPU, < 0.01 ms)
  # ==========================================================================
  if is_structural_table(chunk_text):
    return 3

  # ==========================================================================
  # Tier 2: Holistic Semantic & Epistemic Scope Classifier
  # ==========================================================================
  system_prompt = (
      "You are an expert computational linguist evaluating text chunks for a"
      " localized RAG document slice architecture.\n"
      "Analyze the complete semantic scope, conceptual density, and context"
      " requirements of the provided text chunk.\n\n"
      "Rate the chunk on a scale of 0 to 2:\n"
      "0 (Macro-Synthesis): Broad high-level summary, executive abstract, or"
      " global conclusion. Synthesizes overall research goals, main results, or"
      " systemic takeaways. Requires ZERO surrounding context.\n"
      "1 (Modular Exposition): Independent narrative prose, background"
      " context, hardware setups, or modular definitions. The concepts are"
      " self-contained and easily understood in isolation. Requires MINIMAL"
      " surrounding context.\n"
      "2 (Analytical Deep-Dive): Procedural formulations, mathematical proofs,"
      " experimental discussions, or architectural bottleneck evaluations."
      " While grammatically complete, the reasoning builds directly on ongoing"
      " arguments, formulas, or metrics in the section. Requires MODERATE"
      " surrounding context.\n\n"
      "Instruction: Respond with ONLY the single digit: 0, 1, or 2."
  )

  # Full-length, academically rigorous few-shot exemplars
  few_shot_calibration = (
      "Text:\n"
      "State-of-the-art AI architectures like Retrieval-Augmented Generation"
      " (RAG) often rely on massive data-center infrastructure, creating"
      " barriers to privacy-centric applications. This study introduces a"
      " localized 'Document Slice' framework using a bounded sliding window"
      " instead of full documents. Evaluated on 250 complex queries, our"
      " method achieves a 2.5x preprocessing speedup and a 51% reduction in"
      " energy consumption on an 8GB VRAM GPU with statistically insignificant"
      " impact on retrieval recall.\n"
      "Rating: 0\n\n"
      "Text:\n"
      "For benchmarking, we utilized an Intel Core i7-14650HX CPU paired with"
      " a mobile NVIDIA GeForce RTX 4060 GPU equipped with 8GB GDDR6 VRAM. The"
      " retrieval pipeline integrates LanceDB as a serverless local vector"
      " store, and embedding representations were executed using the"
      " nomic-embed-text-v1.5 model through the Ollama runtime framework.\n"
      "Rating: 1\n\n"
      "Text:\n"
      "During evaluation of the full-context baseline, the Key-Value (KV) cache"
      " footprint vastly exceeded physical memory limits, forcing the runtime"
      " to trigger CUDA Unified Memory paging into host RAM. This induced"
      " extensive page faulting across the PCIe bus and saturated the memory"
      " controller, causing the GPU to enter prolonged I/O stall states rather"
      " than sustaining compute throughput.\n"
      "Rating: 2\n\n"
  )

  user_prompt = f"{few_shot_calibration}Text:\n{chunk_text.strip()}\n\nRating:"

  response = ollama.chat(
      model=model_name,
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt},
      ],
      options={
          "temperature": 0.0,
          "num_predict": 1,  # Single forward-pass argmax
          "top_p": 1.0,
      },
    #   logprobs=True,
    #   top_logprobs=3,  # optional: top 3 alternatives per token
    #   think=False
  )
#   print(f"DEBUG: Raw model output for chunk:\n{response}\n")
  raw_output = response["message"]["content"].strip()
  match = re.search(r"[0-2]", raw_output)
#   print(response.logprobs[0].top_logprobs)
#   for logprob in response.logprobs[0].top_logprobs:
#     print(f"\t {logprob.token}: {logprob.logprob}")
  return int(match.group(0)) if match else 1


# ==============================================================================
# Verbatim Test Suite from citds.tex / citds.pdf
# ==============================================================================
corpus_test_chunks = [
    # --- Category 0: Macro-Synthesis / Global Summaries ---
    {
        "section": "Abstract: Overview & Main Results",
        "expected_k": 0,
        "text": (
            "State-of-the-art AI architectures like"
            ' Retrieval-Augmented Generation (RAG) using LLMs often rely on "Red'
            ' AI" infrastructure, creating significant barriers to'
            " privacy-centric, localized applications. Anthropic’s Contextual"
            " Retrieval addresses chunk-level context loss but introduces"
            " linear computational scaling that exceeds the limits of"
            ' consumer-grade hardware. This study proposes a "Document Slice"'
            " architecture that optimizes context generation by utilizing a"
            " sliding window of neighboring chunks rather than the entire"
            " document. By constraining the context window to a fixed radius,"
            " we facilitate efficient RAG operations on a single GPU with 8GB"
            " of VRAM."
        ),
    },
    {
        "section": "Section IV: Concluding Synthesis",
        "expected_k": 0,
        "text": (
            "This study demonstrates that state-of-the-art contextual"
            " retrieval can be successfully adapted for resource-constrained"
            " environments. By replacing full-document context with a localized"
            " 'Document Slice' sliding window, we have developed an RAG"
            " architecture that fits within the 8GB of VRAM available on"
            " consumer-grade GPUs. The proposed method achieves a 2.5x speedup"
            " and a 51% reduction in energy consumption with a statistically"
            " insignificant impact on retrieval recall. These results have"
            " significant implications for the development of private, local"
            " research assistants and the broader adoption of 'Green AI'"
            " practices."
        ),
    },
    # --- Category 1: Modular Exposition & Infrastructure ---
    {
        "section": "Introduction: Landscape Framing",
        "expected_k": 1,
        "text": (
            "The contemporary technological landscape is dominated by Large"
            " Language Models (LLMs) that have fundamentally altered how we"
            " interact with information. However, a growing 'compute divide'"
            " threatens the democratization of these tools. Much of the"
            " current literature assumes the availability of data-center-grade"
            " infrastructure, a trend dubbed 'Red AI' that prioritizes raw"
            " performance over computational efficiency. This reliance on"
            " massive resources creates a substantial barrier for localized,"
            " privacy-centric applications, particularly for academic teams"
            " and individual researchers."
        ),
    },
    {
        "section": "Section II-E: Hardware Specification",
        "expected_k": 1,
        "text": (
            "For benchmarking, we utilized a GeForce RTX 4060 Mobile GPU (8GB"
            " VRAM) and an Intel Core i7-14650HX CPU. The RAG pipeline"
            " integrated LanceDB for vector storage, and the embedding model"
            " executed in the Ollama inference framework."
        ),
    },
    # --- Category 2: Analytical Deep-Dives & Procedural Mechanics ---
    {
        "section": "Section II-C: Sliding Window Logic",
        "expected_k": 2,
        "text": (
            "For a given chunk i, the document slice is constructed as a"
            " continuous segment from chunk (i - k) to (i + k). This ensures"
            " the target chunk is positioned centrally within the prompt,"
            " mitigating the primacy and recency biases inherent in LLMs. For"
            " document boundaries (start or end), the window is truncated on"
            " one side and extended on the other to maintain a consistent total"
            " token count."
        ),
    },
    {
        "section": "Section II-F: Custom Metric Derivation (MAR)",
        "expected_k": 2,
        "text": (
            "We employed four primary metrics: Recall@K, Mean Reciprocal Rank"
            " (MRR), Mean Average Rank (MAR, our custom metric), and"
            " preprocessing latency. MAR = (1 / |Q|) * sum_q ( (1 / |R_q cap"
            " G_q|) * sum_c rank(c) ) where Q is the set of all queries, R_q"
            " is the set of retrieved documents, and G_q denotes ground-truth"
            " relevant documents."
        ),
    },
    {
        "section": "Section III-E: Memory Profiling Analysis",
        "expected_k": 2,
        "text": (
            "During the evaluation of the Anthropic baseline, a critical"
            " hardware bottleneck was identified. While the quantized weights"
            " of the model natively fit within the 8GB VRAM limit, the"
            " Key-Value (KV) cache footprint for full-document context"
            " sequences vastly exceeded the remaining physical memory. To"
            " prevent Out-Of-Memory (OOM) failures, the inference backend"
            " utilized CUDA Unified Memory, dynamically offloading the KV"
            " cache overflow into the host system RAM."
        ),
    },
    # --- Category 3: Tabular Environments ---
    {
        "section": "Table I: Ablation Benchmark Matrix",
        "expected_k": 3,
        "text": (
            "| Method | Time | Recall@20 | MRR@20 | MAR@20 |\n"
            "| :--- | :--- | :--- | :--- | :--- |\n"
            "| Hybrid retrieval | 0 | 0.9393 | 0.7748 | 4.0363 |\n"
            "| doc slice k=0 | 7m28s | 0.9373 | 0.7861 | 3.8083 |\n"
            "| doc slice k=1 | 12m08s | 0.9433 | 0.7990 | 3.7260 |\n"
            "| doc slice k=2 | 17m28s | 0.9467 | 0.7680 | 3.8095 |\n"
            "| doc slice k=3 | 22m41s | 0.9453 | 0.7877 | 4.0200 |\n"
            "| doc slice k=4 | 28m11s | 0.9433 | 0.8933 | 3.9588 |\n"
            "| Anthropic | 71m34s | 0.9540 | 0.7863 | 3.9355 |"
        ),
    },
    {
        "section": "Table II: Comparative Baseline Summary",
        "expected_k": 3,
        "text": (
            "| Metric | Anthropic | Doc slice | Relative Change |\n"
            "| :--- | :--- | :--- | :--- | :--- |\n"
            "| Recall@20 | 95.4% | 94.5% | -0.9% |\n"
            "| MRR@20 | 0.786 | 0.77 | -2% |\n"
            "| MAR@20 | 3.93 | 4.01 | 1.8% |\n"
            "| Preprocessing Time | 71m34s | 29m09s | -60% |"
        ),
    },
]

# ==============================================================================
# Execution Benchmark
# ==============================================================================
if __name__ == "__main__":
  print(
      f"{'Chunk Origin':<42} | {'Target':<6} | {'Pred':<6} | {'Result':<6} |"
      f" {'Latency':<8}"
  )
  print("-" * 76)

  exact_matches = 0
  latencies = []

  for item in corpus_test_chunks:
    t0 = time.perf_counter()
    k_pred = predict_dynamic_k(item["text"])
    elapsed_ms = (time.perf_counter() - t0) * 1000.0
    latencies.append(elapsed_ms)

    k_gold = item["expected_k"]
    matched = k_pred == k_gold
    if matched:
      exact_matches += 1

    status = "PASS" if matched else "FAIL"
    print(
        f"{item['section']:<42} | k={k_gold:<4} | k={k_pred:<4} | {status:<6}"
        f" | {elapsed_ms:6.1f} ms"
    )

  total = len(corpus_test_chunks)
  print("-" * 76)
  print(
      f"Strict Exact Match Accuracy: {exact_matches}/{total}"
      f" ({exact_matches/total*100:.1f}%)"
  )
  print(f"Mean Decision Latency:     {sum(latencies)/total:.2f} ms per chunk")

Chunk Origin                               | Target | Pred   | Result | Latency 
----------------------------------------------------------------------------
Abstract: Overview & Main Results          | k=0    | k=1    | FAIL   |  167.2 ms
Section IV: Concluding Synthesis           | k=0    | k=0    | PASS   |  157.0 ms
Introduction: Landscape Framing            | k=1    | k=1    | PASS   |  157.6 ms
Section II-E: Hardware Specification       | k=1    | k=1    | PASS   |  148.6 ms
Section II-C: Sliding Window Logic         | k=2    | k=2    | PASS   |  155.2 ms
Section II-F: Custom Metric Derivation (MAR) | k=2    | k=2    | PASS   |  163.1 ms
Section III-E: Memory Profiling Analysis   | k=2    | k=2    | PASS   |  164.6 ms
Table I: Ablation Benchmark Matrix         | k=3    | k=3    | PASS   |    0.0 ms
Table II: Comparative Baseline Summary     | k=3    | k=3    | PASS   |    0.0 ms
----------------------------------------------------------------------------
Strict Exact Match Accura

In [1]:
from docling.document_converter import DocumentConverter
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling.chunking import HybridChunker
from transformers import AutoTokenizer

pdf_path = "input/A_Hybrid_Gaze_Distance_Estimation_via_Cross-Reference_of_Vergence_and_Depth.pdf"

converter = DocumentConverter()
result = converter.convert(pdf_path)

tokenizer = HuggingFaceTokenizer(
			tokenizer=AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5"),
			max_tokens=2000
		)

chunker = HybridChunker(
            tokenizer=tokenizer,
            merge_peers=True  # Optional, defaults to true
        )

chunks = list(chunker.chunk(dl_doc=result.document))
joined_chunks = "\n\n".join([chunk.text for chunk in chunks])

with open("deleteme.md", "w", encoding="utf-8") as file:
	file.write(joined_chunks)

print("Saved converted Markdown to deleteme.md")

/home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Preset 'granite_vision_v4' already registered for ChartExtractionVlmEngineOptions
[INFO] 2026-09-21 16:02:53,242 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-21 16:02:53,243 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-09-21 16:02:53,249 [RapidOCR] download_file.py:60: File exists and is valid: /home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.12/site-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-09-21 16:02:53,250 [RapidOCR] main.py:50: Using /home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.12/site-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-09-21 16:02:5

Saved converted Markdown to deleteme.md


In [7]:
from docling_core.types.doc import DocItemLabel

# Deterministic AST inspection (Zero CPU overhead, zero false positives)
for chunk in chunks:
	has_table = any(item.label == DocItemLabel.TABLE for item in chunk.meta.doc_items)
	print(f"Has Table: {has_table}")
	print(f"\t Chunk headings: {chunk.meta.headings}")
	if "To deliver an optimal Mixed Reality (MR) experience, wherein virtual " in chunk.text:
		print(f"Chunk Text:\n{chunk.text}\n")
	if has_table:
		print(f"Chunk Text:\n{chunk.text}\n")

Has Table: False
	 Chunk headings: None
Has Table: False
	 Chunk headings: ['DAE-YONG CHO 1, 2 AND MIN-KOO KANG 1, 2 (Member, IEEE)']
Chunk Text:
1 Korea Institute of Science and Technology, Seoul, South Korea
2 KHU-KIST Department of Converging Science and Technology, Kyung Hee University, Seoul, South Korea
Corresponding author: Min-Koo Kang (e-mail: minkoo@kist.re.kr).
This work was financially supported by the Institute of Civil-Military Technology Cooperation Program funded by the Defense Acquisition Program Administration and Ministry of Trade, Industry and Energy of Korean government (grant No. 17-CM-DP-29).
ABSTRACT To deliver an optimal Mixed Reality (MR) experience, wherein virtual elements and real-world objects are seamlessly merged, it is vital to ensure a consistent vergence-accommodation distance. This necessitates the advancement of technology to precisely estimate the user's gaze distance. Presently, various MR devices employ small eye-tracking cameras to capture both 

In [ ]:
import time
import re
import ollama
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling_core.types.doc import DocItemLabel
import unicodedata
from transformers import AutoTokenizer

OLLAMA_MODEL_NAME = "qwen3.5:4b"
pdf_path = "input/A_Hybrid_Gaze_Distance_Estimation_via_Cross-Reference_of_Vergence_and_Depth.pdf"


def clean_docling_chunk_string(chunk):
    
	# 2️⃣ Normalize Unicode and replace problematic punctuation
	chunk = unicodedata.normalize("NFKD", chunk).replace("\u00A0", " ")
	chunk = chunk.translate(str.maketrans({
		"–": "-", "—": "-", "‘": "'", "’": "'", "“": '"', "”": '"'
	}))

	# 3️⃣ Remove URLs (massive tokenizers killers)
	chunk = re.sub(r"http\S+", "", chunk)

	# 4️⃣ Normalize whitespace but preserve paragraphs
	chunk = re.sub(r"[ \t]+", " ", chunk)
	chunk = re.sub(r"\n\s*\n", "\n\n", chunk)  # merge single newlines, keep double
	chunk = chunk.strip()

	return chunk


def classify_prose_with_slm(chunk_text: str, model_name: str = OLLAMA_MODEL_NAME) -> int:
    """
    Tier 2: Single-token argmax classifier for narrative prose across k in {0, 1, 2}.
    """
	system_prompt = (
			"You are an expert computational linguist evaluating text chunks for a"
			" localized RAG document slice architecture.\n"
			"Analyze the complete semantic scope, conceptual density, and context"
			" requirements of the provided text chunk.\n\n"
			"Rate the chunk on a scale of 0 to 2:\n"
			"0 (Macro-Synthesis): Broad high-level summary, executive abstract, or"
			" global conclusion. Synthesizes overall research goals, main results, or"
			" systemic takeaways. Requires ZERO surrounding context.\n"
			"1 (Modular Exposition): Independent narrative prose, background"
			" context, hardware setups, or modular definitions. The concepts are"
			" self-contained and easily understood in isolation. Requires MINIMAL"
			" surrounding context.\n"
			"2 (Analytical Deep-Dive): Procedural formulations, mathematical proofs,"
			" experimental discussions, or architectural bottleneck evaluations."
			" While grammatically complete, the reasoning builds directly on ongoing"
			" arguments, formulas, or metrics in the section. Requires MODERATE"
			" surrounding context.\n\n"
			"Instruction: Respond with ONLY the single digit: 0, 1, or 2."
		)
 
	few_shot_calibration = (
			"Text:\n"
			"State-of-the-art AI architectures like Retrieval-Augmented Generation"
			" (RAG) often rely on massive data-center infrastructure, creating"
			" barriers to privacy-centric applications. This study introduces a"
			" localized 'Document Slice' framework using a bounded sliding window"
			" instead of full documents. Evaluated on 250 complex queries, our"
			" method achieves a 2.5x preprocessing speedup and a 51% reduction in"
			" energy consumption on an 8GB VRAM GPU with statistically insignificant"
			" impact on retrieval recall.\n"
			"Rating: 0\n\n"
			"Text:\n"
			"For benchmarking, we utilized an Intel Core i7-14650HX CPU paired with"
			" a mobile NVIDIA GeForce RTX 4060 GPU equipped with 8GB GDDR6 VRAM. The"
			" retrieval pipeline integrates LanceDB as a serverless local vector"
			" store, and embedding representations were executed using the"
			" nomic-embed-text-v1.5 model through the Ollama runtime framework.\n"
			"Rating: 1\n\n"
			"Text:\n"
			"During evaluation of the full-context baseline, the Key-Value (KV) cache"
			" footprint vastly exceeded physical memory limits, forcing the runtime"
			" to trigger CUDA Unified Memory paging into host RAM. This induced"
			" extensive page faulting across the PCIe bus and saturated the memory"
			" controller, causing the GPU to enter prolonged I/O stall states rather"
			" than sustaining compute throughput.\n"
			"Rating: 2\n\n"
		)

    user_prompt = f"{few_shot_calibration}Text:\n{chunk_text.strip()[:1500]}\n\nRating:"

    response = ollama.chat(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        options={
            "temperature": 0.0,
            "num_predict": 1,
            "top_p": 1.0
        }
    )

    raw_output = response["message"]["content"].strip()
    match = re.search(r"[0-2]", raw_output)
    return int(match.group(0)) if match else 1


def route_docling_chunk(chunk) -> int:
    """
    Assigns context radius k in {0, 1, 2, 3} using Docling AST metadata and SLM gating.
    """
    # =========================================================================
    # TIER 1: Deterministic AST Metadata Gating (Host CPU, < 1 microsecond)
    # =========================================================================
    # 1. Check for Tables in Docling AST
    is_table = any(item.label == DocItemLabel.TABLE for item in chunk.meta.doc_items)
    if is_table:
        return 3

    # 2. Check for Structural Headings (Abstract / Conclusion)
    print(chunk.meta.headings)
    if chunk.meta.headings:
        headings = [h.lower() for h in chunk.meta.headings]
        if any("abstract" in h for h in headings) and len(chunk.text.split()) < 500:
            return 0

    # =========================================================================
    # TIER 2: Semantic Coreference Classifier (Local SLM for Prose)
    # =========================================================================
    return classify_prose_with_slm(clean_docling_chunk_string(chunk.text))


# =============================================================================
# Pipeline Execution
# =============================================================================
if __name__ == "__main__":

    print("Converting document via Docling...")
    converter = DocumentConverter()
    result = converter.convert(pdf_path)

    tokenizer = HuggingFaceTokenizer(
        tokenizer=AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5"),
        max_tokens=2000
    )

    chunker = HybridChunker(tokenizer=tokenizer, merge_peers=True)
    chunks = list(chunker.chunk(dl_doc=result.document))

    print(f"\nProcessing {len(chunks)} Chunks Through the Hybrid Context Router:")
    print(f"{'Chunk ID':<10} | {'Type':<18} | {'Assigned k':<10} | {'Sample Snippet':<45}")
    print("-" * 90)

    stats = {0: 0, 1: 0, 2: 0, 3: 0}
    t_start = time.perf_counter()

    for idx, c in enumerate(chunks):
        is_tab = any(item.label == DocItemLabel.TABLE for item in c.meta.doc_items)
        chunk_type = "Table (AST)" if is_tab else "Prose (SLM)"
        
        k = route_docling_chunk(c)
        stats[k] += 1
        
        clean_snippet = clean_docling_chunk_string(c.text).replace("\n", " ")[:42] + "..."
        print(f"Chunk {idx:<4} | {chunk_type:<18} | k = {k:<6} | {clean_snippet:<45}")

    total_time = time.perf_counter() - t_start
    print("-" * 90)
    print(f"Processed {len(chunks)} chunks in {total_time:.2f}s ({total_time/len(chunks)*1000:.1f} ms/chunk)")
    print(f"Radius Distribution: k=0: {stats[0]} | k=1: {stats[1]} | k=2: {stats[2]} | k=3: {stats[3]}")
    
    k_bar = sum(k * count for k, count in stats.items()) / len(chunks)
    print(f"Effective Mean Radius (k̄): {k_bar:.2f} (Compared to static baseline k=3)")

/home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Preset 'granite_vision_v4' already registered for ChartExtractionVlmEngineOptions


Converting document via Docling...


[INFO] 2026-09-21 16:58:23,320 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-21 16:58:23,322 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-09-21 16:58:23,329 [RapidOCR] download_file.py:60: File exists and is valid: /home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.12/site-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-09-21 16:58:23,330 [RapidOCR] main.py:50: Using /home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.12/site-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-09-21 16:58:23,609 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-21 16:58:23,609 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-09-21 16:58:23,610 [RapidOCR] download_file.py:60: File exists and is valid: /home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[


Processing 14 Chunks Through the Hybrid Context Router:
Chunk ID   | Type               | Assigned k | Sample Snippet                               
------------------------------------------------------------------------------------------
None
Chunk 0    | Prose (SLM)        | k = 1      | Date of publication xxxx 00, 0000, date of...
['DAE-YONG CHO 1, 2 AND MIN-KOO KANG 1, 2 (Member, IEEE)']
Chunk 1    | Prose (SLM)        | k = 1      | 1 Korea Institute of Science and Technolog...
['I. INTRODUCTION']
Chunk 2    | Prose (SLM)        | k = 1      | Extended Reality (XR) technologies, includ...
['A. GAZE DISTANCE FROM VERGENCE']
Chunk 3    | Prose (SLM)        | k = 1      | The vergence has been regarded as one of t...
['B. 3D GAZE ESTIMATION WITH RGB-D CAMERA']
Chunk 4    | Prose (SLM)        | k = 1      | Though the depth cue from an active sensor...
['III. PROPOSED METHOD']
Chunk 5    | Prose (SLM)        | k = 1      | In this section, we explain a hybrid metho...
['A. GAZE DIS